# unbroadcast-pattern — worked example 3: Unbroadcast wired into add_back

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `unbroadcast-pattern`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The full unbroadcast does both passes in order: peel leading axes, then collapse expanded size-1 axes. The binary backward `add_back` for `out = x + y` has local gradient 1, so it returns `grad_out` unbroadcast to each input's shape — bridging the elementwise math gradient and the input-shape contract.

## Worked solution

We build the complete unbroadcast and use it in an `add` backward.

1. `unbroadcast` peels leading axes first, then collapses size-1 axes with `keepdim=True`. Order matters: peeling first guarantees the axis indices in the second pass line up with `original`.
2. For `out = x + y`, the derivative w.r.t. each input is 1, so the local gradient is just `grad_out`; we unbroadcast it down to that input's shape.
3. We test the combined case: `x` shape `(1, 4)`, `y` shape `(2, 3, 4)`. `grad_x` must come back `(1, 4)` and `grad_y` stays `(2, 3, 4)`.

The printed shapes confirm each gradient matches its input.

In [ ]:
import torch as t

def unbroadcast(grad: t.Tensor, original: t.Tensor) -> t.Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def add_back0(grad_out, out, x, y):
    return unbroadcast(grad_out, x)

def add_back1(grad_out, out, x, y):
    return unbroadcast(grad_out, y)

x = t.zeros(1, 4)
y = t.zeros(2, 3, 4)
grad_out = t.ones(2, 3, 4)
gx = add_back0(grad_out, None, x, y)
gy = add_back1(grad_out, None, x, y)
print('grad_x shape:', tuple(gx.shape))
print('grad_y shape:', tuple(gy.shape))